# Variable Elimination

**Companion wiki page:** https://ml-viz-ruby.vercel.app/wiki/variable-elimination

Exact inference in a Bayesian Network, from scratch: factors as NumPy tables, factor multiplication, summing out variables, and the full query $P(\text{Wet} \mid \text{Rain}=T)$ on the Sprinkler network.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2d3148'
plt.rcParams['grid.color'] = '#2d3148'
np.random.seed(42)

## Factors as dictionaries of tables

A **factor** is a function over a set of variables, stored as an n-dimensional table. We represent one as `(vars, table)` where `vars` is a tuple of variable names and `table` has one axis per variable (all binary here: index 0 = False, 1 = True).

In [ ]:
# Sprinkler network: Rain -> Sprinkler, Rain -> Wet, Sprinkler -> Wet
# P(Rain)
f_rain = (("Rain",), np.array([0.8, 0.2]))

# P(Sprinkler | Rain): axes (Sprinkler, Rain)
f_sprinkler = (("Sprinkler", "Rain"), np.array([[0.6, 0.99],
                                                [0.4, 0.01]]))

# P(Wet | Sprinkler, Rain): axes (Wet, Sprinkler, Rain)
f_wet = (("Wet", "Sprinkler", "Rain"), np.array([
    [[1.00, 0.20], [0.10, 0.05]],   # Wet = F
    [[0.00, 0.80], [0.90, 0.95]],   # Wet = T
]))

factors = [f_rain, f_sprinkler, f_wet]
for v, t in factors:
    print(v, "table sums:", t.sum())

## Factor operations

- **Restrict** — slice the table at the observed value (set evidence).
- **Multiply** — align variables with broadcasting, multiply tables.
- **Sum out** — sum the table along one variable's axis.

In [ ]:
def restrict(factor, var, value):
    vars_, table = factor
    if var not in vars_:
        return factor
    ax = vars_.index(var)
    new_table = np.take(table, value, axis=ax)
    new_vars = tuple(v for v in vars_ if v != var)
    return (new_vars, new_table)

def multiply(f1, f2):
    v1, t1 = f1
    v2, t2 = f2
    all_vars = tuple(dict.fromkeys(v1 + v2))  # ordered union
    def expand(vars_, table):
        # add singleton axes for missing vars, then transpose to all_vars order
        for v in all_vars:
            if v not in vars_:
                table = table[..., None]
                vars_ = vars_ + (v,)
        perm = [vars_.index(v) for v in all_vars]
        return np.transpose(table, perm)
    return (all_vars, expand(v1, t1) * expand(v2, t2))

def sum_out(factor, var):
    vars_, table = factor
    ax = vars_.index(var)
    return (tuple(v for v in vars_ if v != var), table.sum(axis=ax))

## The elimination loop

To compute $P(Q \mid E=e)$:

1. Restrict every factor on the evidence.
2. For each hidden variable: multiply the factors that mention it, sum it out.
3. Multiply what remains, normalize.

In [ ]:
def variable_elimination(factors, query, evidence, order):
    # 1. set evidence
    fs = list(factors)
    for var, val in evidence.items():
        fs = [restrict(f, var, val) for f in fs]
    # 2. eliminate hidden variables in order
    for z in order:
        mentions = [f for f in fs if z in f[0]]
        rest = [f for f in fs if z not in f[0]]
        if not mentions:
            continue
        prod = mentions[0]
        for f in mentions[1:]:
            prod = multiply(prod, f)
        fs = rest + [sum_out(prod, z)]
    # 3. multiply remaining factors, normalize
    result = fs[0]
    for f in fs[1:]:
        result = multiply(result, f)
    vars_, table = result
    # marginalize anything that isn't the query (e.g. leftover constants)
    for v in list(vars_):
        if v != query:
            result = sum_out(result, v)
            vars_, table = result
    return result[1] / result[1].sum()

p = variable_elimination(factors, query="Wet", evidence={"Rain": 1}, order=["Sprinkler"])
print(f"P(Wet | Rain=T) = [F: {p[0]:.4f}, T: {p[1]:.4f}]")

## Sanity check against the joint

The whole point of variable elimination is to avoid building the full joint — but on 3 binary variables we *can* build it and verify.

In [ ]:
# joint[w, s, r] = P(r) P(s|r) P(w|s,r)
joint = (f_rain[1][None, None, :]
         * f_sprinkler[1][None, :, :]
         * f_wet[1])
print("joint sums to:", joint.sum().round(6))

cond = joint[:, :, 1].sum(axis=1)         # fix Rain=T, sum over Sprinkler
cond = cond / cond.sum()
print("brute force P(Wet | Rain=T):", cond.round(4))
assert np.allclose(cond, p), "VE must match brute force"
print("matches variable elimination ✓")

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise — query the other direction

**Recap:** the same procedure answers any conditional query. Diagnostic reasoning runs *against* the arrows: given that the grass is wet, was it raining?

Compute $P(\text{Rain} \mid \text{Wet}=T)$ using `variable_elimination` (eliminate Sprinkler).

In [ ]:
def p_rain_given_wet():
    # TODO(you): call variable_elimination with the right query,
    # evidence, and elimination order; return the length-2 array.
    ...

p_rain = p_rain_given_wet()
p_rain

In [ ]:
# Run me — passes silently when correct
joint = (f_rain[1][None, None, :] * f_sprinkler[1][None, :, :] * f_wet[1])
expected = joint[1].sum(axis=0); expected = expected / expected.sum()
assert p_rain is not None and not isinstance(p_rain, type(Ellipsis)), "fill in the TODO first"
assert np.allclose(np.asarray(p_rain), expected, atol=1e-8)
print()

<details>
<summary>Solution</summary>

```python
def p_rain_given_wet():
    return variable_elimination(
        factors, query="Rain", evidence={"Wet": 1}, order=["Sprinkler"]
    )
```
</details>